In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load the training data

train = pd.read_csv('../data/final_train.csv')

# Convert Date back to datetime
train['Date'] = pd.to_datetime(train['Date'])

In [4]:
# Extract Year, Month, and Week from Date

train['Year'] = train['Date'].dt.year
train['Month'] = train['Date'].dt.month
train['Week'] = train['Date'].dt.isocalendar().week.astype(int)

In [5]:
# Split the data into training and validation sets based on Date

split_date = train['Date'].quantile(0.8)

train_df = train[train['Date'] <= split_date]
val_df = train[train['Date'] > split_date]

features_to_drop = ['Weekly_Sales', 'Date']

X_train = train_df.drop(columns=features_to_drop)
y_train = train_df['Weekly_Sales']

X_val = val_df.drop(columns=features_to_drop)
y_val = val_df['Weekly_Sales']

In [6]:
# One-hot encode the 'Type' column

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded_train = encoder.fit_transform(X_train[['Type']])
encoded_val = encoder.transform(X_val[['Type']])

encoded_cols = encoder.get_feature_names_out(['Type'])

# Drop original column
X_train = X_train.drop('Type', axis=1).reset_index(drop=True)
X_val = X_val.drop('Type', axis=1).reset_index(drop=True)

# Add encoded columns
X_train = pd.concat([X_train, pd.DataFrame(encoded_train, columns=encoded_cols)], axis=1)
X_val = pd.concat([X_val, pd.DataFrame(encoded_val, columns=encoded_cols)], axis=1)

In [7]:
# Scale the features using RobustScaler

scaler = RobustScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [8]:
# Train the models

models = {
    "Linear_Regression": LinearRegression(),
    "Random_Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        objective='reg:squarederror'
    )
}

results = {}
trained_models = {}

for name, model in models.items():
    print(f"\nTraining {name}...")

    if name == "Linear_Regression":
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_val_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_val)

    mae = mean_absolute_error(y_val, preds)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    r2 = r2_score(y_val, preds)

    results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2}
    trained_models[name] = model

    print(f"{name} → MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.4f}")


Training Linear_Regression...
Linear_Regression → MAE: 14542.03, RMSE: 20898.35, R2: 0.0950

Training Random_Forest...
Random_Forest → MAE: 1813.39, RMSE: 3764.04, R2: 0.9706

Training XGBoost...
XGBoost → MAE: 3459.77, RMSE: 5676.10, R2: 0.9332


In [9]:
# display results in a DataFrame sorted by RMSE

results_df = pd.DataFrame(results).T.sort_values(by="RMSE")

print("Model Performance:")
print(results_df)

best_model_name = results_df.index[0]
best_model = trained_models[best_model_name]

print(f"\nBest Model: {best_model_name}")


Model Performance:
                            MAE          RMSE        R2
Random_Forest       1813.386660   3764.036782  0.970640
XGBoost             3459.770830   5676.103180  0.933235
Linear_Regression  14542.033034  20898.351235  0.094952

Best Model: Random_Forest


In [10]:
# Now we will compute feature importance for the Random Forest model using permutation importance

In [11]:
print("\nCalculating Permutation Importance...")

perm = permutation_importance(
    best_model,
    X_val,  # Use unscaled features for permutation importance because the best_model was trained on unscaled features
    y_val,  # Use unscaled target for permutation importance because the best_model was trained on unscaled target
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    'Feature': X_val.columns,
    'Importance': perm.importances_mean
}).sort_values(by='Importance', ascending=False)

print("\nTop Features:\n")
print(importance_df.head(15))



Calculating Permutation Importance...

Top Features:

         Feature  Importance
1           Dept    1.833649
3           Size    0.590223
0          Store    0.130030
11           CPI    0.064615
17        Type_B    0.028849
15          Week    0.023215
12  Unemployment    0.017503
16        Type_A    0.010514
4    Temperature    0.007070
18        Type_C    0.000999
9      MarkDown4    0.000545
8      MarkDown3    0.000373
2      IsHoliday    0.000212
10     MarkDown5    0.000193
14         Month    0.000139


In [12]:
top_features = [
    'Dept',
    'Size',
    'Store',
    'CPI',
    'Week',
    'Unemployment',
    'Type'   # categorical (will be encoded)
]

In [13]:
train = pd.read_csv('../data/final_train.csv')
train['Date'] = pd.to_datetime(train['Date'])

In [14]:
train['Year'] = train['Date'].dt.year
train['Month'] = train['Date'].dt.month
train['Week'] = train['Date'].dt.isocalendar().week.astype(int)

In [15]:
# split the data with top features only
split_date = train['Date'].quantile(0.8)

train_df = train[train['Date'] <= split_date]
val_df = train[train['Date'] > split_date]

X_train = train_df[top_features]
y_train = train_df['Weekly_Sales']

X_val = val_df[top_features]
y_val = val_df['Weekly_Sales']

In [16]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded_train = encoder.fit_transform(X_train[['Type']])
encoded_val = encoder.transform(X_val[['Type']])

encoded_cols = encoder.get_feature_names_out(['Type'])

# Drop original column
X_train = X_train.drop('Type', axis=1).reset_index(drop=True)
X_val = X_val.drop('Type', axis=1).reset_index(drop=True)

# Add encoded columns
X_train = pd.concat([X_train, pd.DataFrame(encoded_train, columns=encoded_cols)], axis=1)
X_val = pd.concat([X_val, pd.DataFrame(encoded_val, columns=encoded_cols)], axis=1)

In [17]:
# Train a new Random Forest model on top features only
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

print("Training model on top features...")
model.fit(X_train, y_train)


Training model on top features...


,n_estimators,200
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [18]:
# Evaluate the model on the validation set
preds = model.predict(X_val)

mae = mean_absolute_error(y_val, preds)
rmse = np.sqrt(mean_squared_error(y_val, preds))
r2 = r2_score(y_val, preds)

print("\nPerformance (Top Features Model):")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {r2:.4f}")


Performance (Top Features Model):
MAE: 1781.82
RMSE: 3732.26
R2: 0.9711


In [19]:
# only run this cell when you you want to save models locally
# also uncomment the model loading link in api.py to load the local model instead of the google drive link
# Save the top-features model and artifacts
joblib.dump(model, '../models/best_model_top.pkl', compress=9)
joblib.dump(encoder, '../models/encoder_top.pkl')
joblib.dump(X_train.columns.tolist(), '../models/model_top_features.pkl')
joblib.dump(top_features, '../models/raw_top_features.pkl')

print("\nTop-features model saved successfully!")


Top-features model saved successfully!
